# Brick 2 — Single-agent Gymnasium environment (deep dive)

A thorough exploration of `FisheryEnv`: API contract, reward shape, stochastic variance, action sweep, physical cap mechanics, and sensitivity to the environment's `max_harvest_rate`.

**Objectives**:
1. Verify the API contract.
2. Visualize the reward function and *why* concavity matters.
3. See the variance of a random fisher across multiple seeds.
4. Sweep the action space to see the sustainability vs profit trade-off (the tragedy of the commons).
5. Inspect the physical cap and stock recovery.
6. Watch how `max_harvest_rate` shifts the optimal policy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from bilevel_fishery.ecology import EcologyParams
from bilevel_fishery.envs import FisheryEnv

plt.rcParams["figure.figsize"] = (10, 5)

# Demo settings. At default L-V parameters the natural period is
# 2π/√(αγ) ≈ 5.13 time units. With dt=0.05 and horizon=400 each episode
# covers 20 time units (~4 periods). Long enough that unsustainable
# (greedy) actions clearly collapse the stock and the cap shuts down the
# harvest, making the tragedy of the commons unambiguous.
DEMO_DT = 0.05
DEMO_HORIZON = 400
DEMO_PARAMS = EcologyParams(dt=DEMO_DT)

# Constant harvest H > 0 turns the L-V centre into an unstable spiral, so
# orbits grow until the cap engages and the env catches the resulting near-
# zero biomass at the env layer (fish→0). The y-axis on trajectory plots
# is clamped to [-2, FISH_PLOT_YMAX] to keep the visualization readable;
# the underlying dynamics still drive the simulation faithfully.
FISH_PLOT_YMAX = 30


def make_env(**kwargs):
    """FisheryEnv with the demo dt; override params/horizon via kwargs."""
    kwargs.setdefault("params", DEMO_PARAMS)
    kwargs.setdefault("horizon", DEMO_HORIZON)
    return FisheryEnv(**kwargs)

## 1. Sanity check: reset + step

The API contract: `reset(seed)` returns `(obs, info)`, `step(action)` returns `(obs, reward, terminated, truncated, info)`.

In [ ]:
env = make_env()
obs, info = env.reset(seed=42)
print(f"obs shape={obs.shape}, dtype={obs.dtype}")
print(f"obs (fish_norm, algae_norm) = ({obs[0]:.3f}, {obs[1]:.3f})")
print(f"action_space = {env.action_space}")
print(f"observation_space = {env.observation_space}")
print(f"dt = {env.params.dt}, horizon = {env.horizon}")
total_time = env.params.dt * env.horizon
lv_period = 2 * np.pi / np.sqrt(env.params.alpha * env.params.gamma)
print(
    f"Episode covers {total_time} time units "
    f"(~{total_time / lv_period:.1f} L-V periods)."
)

next_obs, reward, terminated, truncated, info = env.step(
    np.array([0.5], dtype=np.float32)
)
print("\nAfter one step at action=0.5:")
print(f"  next obs = ({next_obs[0]:.3f}, {next_obs[1]:.3f})")
print(f"  reward   = {reward:.4f}")
print(f"  terminated = {terminated}, truncated = {truncated}")
print(f"  info     = {info}")

## 2. The reward function: why concavity?

[D-001](../docs/decisions/D-001-reward-function.md) selected `r = log(1 + harvest)` over a linear reward and over the master codebase's utility-scaled reward. Here is what those three curves look like.

The selected curve is **concave**: doubling the catch does *not* double the reward. This bakes risk aversion into the learning signal.

In [ ]:
h = np.linspace(0, 2.0, 200)
fish_norm = 0.5  # mid-range stock for the utility-scaling option

label_b = (
    r"utility scaling, master (Option B): "
    r"$r = h \cdot \mathrm{fish}/\mathrm{max}$"
)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(h, h, label=r"linear (Option A): $r = h$", lw=2)
ax.plot(h, h * fish_norm, label=label_b, lw=2)
ax.plot(
    h,
    np.log1p(h),
    label=r"log1p — selected, D-001 (Option C): $r = \log(1 + h)$",
    lw=2.5,
    color="C2",
)
ax.set_xlabel("harvest_realized")
ax.set_ylabel("reward")
ax.set_title("Reward function shape — concavity creates diminishing marginal returns")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
ax.annotate(
    "large h gets less reward per unit\n→ agent learns to spread catches",
    xy=(1.6, np.log1p(1.6)),
    xytext=(1.2, 1.8),
    arrowprops={"arrowstyle": "->", "color": "C2", "alpha": 0.6},
    fontsize=10,
)
plt.tight_layout()
plt.show()

## 3. Random fisher — stochastic variance over 30 seeds

A single trajectory can be misleading: the initial state has log-normal noise (`noise_std=0.05`), which propagates through the dynamics. Running 30 seeds shows the **distribution** of outcomes.

We show mean ± 1σ bands for fish biomass, algae biomass, instantaneous reward, and cumulative reward over 20 time units (~4 L-V periods).

In [ ]:
N_SEEDS = 30

fish_arr = np.zeros((N_SEEDS, DEMO_HORIZON))
algae_arr = np.zeros((N_SEEDS, DEMO_HORIZON))
reward_arr = np.zeros((N_SEEDS, DEMO_HORIZON))
cumrew_arr = np.zeros((N_SEEDS, DEMO_HORIZON))

for s in range(N_SEEDS):
    env = make_env()
    env.reset(seed=s)
    rng = np.random.default_rng(s)
    cum = 0.0
    for t in range(DEMO_HORIZON):
        action = rng.uniform(0, 1, size=(1,)).astype(np.float32)
        _, reward, _, _, info = env.step(action)
        fish_arr[s, t] = info["fish"]
        algae_arr[s, t] = info["algae"]
        reward_arr[s, t] = reward
        cum += reward
        cumrew_arr[s, t] = cum

time_axis = np.arange(DEMO_HORIZON) * DEMO_DT


def band(ax, data, label, color, clip=None):
    """Mean line + ±1σ shaded band over time, optionally clipping y range."""
    mu = data.mean(axis=0)
    sd = data.std(axis=0)
    ax.plot(time_axis, mu, color=color, label=f"{label} (mean)")
    ax.fill_between(time_axis, mu - sd, mu + sd, alpha=0.25, color=color, label="±1σ")
    if clip is not None:
        ax.set_ylim(*clip)
    ax.legend(fontsize=9)


fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
band(axes[0, 0], fish_arr, "fish", "C0", clip=(-2, FISH_PLOT_YMAX))
band(axes[0, 1], algae_arr, "algae", "C1", clip=(0, 80))
band(axes[1, 0], reward_arr, "instant reward", "C3")
band(axes[1, 1], cumrew_arr, "cumulative reward", "C2")
axes[0, 0].set_ylabel("biomass")
axes[1, 0].set_ylabel("instant reward")
axes[1, 1].set_ylabel("cumulative reward")
axes[1, 0].set_xlabel("time")
axes[1, 1].set_xlabel("time")
fig.suptitle(f"Random fisher, {N_SEEDS} seeds — mean ± 1σ over time")
plt.tight_layout()
plt.show()

print(
    f"Cumulative reward at t={DEMO_HORIZON * DEMO_DT:.1f}: "
    f"{cumrew_arr[:, -1].mean():.2f} ± {cumrew_arr[:, -1].std():.2f}"
)

## 4. Action sweep — the sustainability vs profit trade-off

We run 21 fishers, each with a **constant action** `a ∈ {0.00, 0.05, ..., 1.00}`, on the same seed. Each episode lasts 20 time units (~4 L-V periods). We track:

- the fish biomass over time (one trajectory per action),
- the **final stock** (sustainability proxy),
- the **cumulative reward** (profit proxy),
- the Pareto front of the two (action color-coded),
- a heatmap of fish biomass `action × time`.

This is **the tragedy of the commons** in five figures: greedy actions drive the stock so low that the physical cap eventually shuts the harvest down.

In [ ]:
actions = np.linspace(0.0, 1.0, 21)

fish_history = np.zeros((len(actions), DEMO_HORIZON))
finals = np.zeros(len(actions))
cumrews = np.zeros(len(actions))

for i, a in enumerate(actions):
    env = make_env()
    env.reset(seed=42)
    action_vec = np.array([a], dtype=np.float32)
    cum = 0.0
    for t in range(DEMO_HORIZON):
        _, reward, _, _, info = env.step(action_vec)
        fish_history[i, t] = info["fish"]
        cum += reward
    finals[i] = info["fish"]
    cumrews[i] = cum

i_best = int(np.argmax(cumrews))
print(f"Swept {len(actions)} actions × {DEMO_HORIZON} steps (dt={DEMO_DT}).")
print(f"Total simulated time per episode: {DEMO_HORIZON * DEMO_DT} time units.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
cmap = plt.get_cmap("viridis")
for i, a in enumerate(actions):
    ax.plot(time_axis, fish_history[i], color=cmap(a), alpha=0.9, lw=1.4)
ax.set_xlabel("time")
ax.set_ylabel("fish biomass")
ax.set_title("Fish trajectories under constant action — color = action intensity")
ax.set_ylim(-2, FISH_PLOT_YMAX)
ax.axhline(
    DEMO_PARAMS.alpha / DEMO_PARAMS.beta,
    color="green",
    ls=":",
    lw=2,
    alpha=0.8,
    label=r"natural equilibrium $F^* = \alpha/\beta$",
)
ax.legend(loc="upper right")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])
plt.colorbar(sm, ax=ax, label="constant action a")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(actions, finals, width=0.04, color="C0", alpha=0.85, edgecolor="k")
axes[0].set_xlabel("constant action a")
axes[0].set_ylabel("fish biomass at end of episode")
axes[0].set_title("Final stock vs action — sustainability")
axes[0].axhline(
    DEMO_PARAMS.alpha / DEMO_PARAMS.beta,
    color="green",
    ls=":",
    alpha=0.7,
    label=r"natural equilibrium $F^*$",
)
axes[0].legend()

axes[1].bar(actions, cumrews, width=0.04, color="C3", alpha=0.85, edgecolor="k")
axes[1].set_xlabel("constant action a")
axes[1].set_ylabel("cumulative reward")
axes[1].set_title("Total profit over the episode")
axes[1].axvline(actions[i_best], color="orange", ls="--", alpha=0.8)
axes[1].annotate(
    f"best a* = {actions[i_best]:.2f}",
    xy=(actions[i_best], cumrews[i_best]),
    xytext=(actions[i_best] + 0.05, cumrews[i_best] * 0.55),
    arrowprops={"arrowstyle": "->", "color": "orange"},
    fontsize=10,
)
plt.tight_layout()
plt.show()

print(f"Optimal action: a* = {actions[i_best]:.2f}")
print(f"  → final stock at a*: {finals[i_best]:.2f}")
print(f"  → final stock at a=1.0 (greediest): {finals[-1]:.2f}")
print(f"  → cumulative reward at a*: {cumrews[i_best]:.2f}")
print(f"  → cumulative reward at a=1.0: {cumrews[-1]:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    finals,
    cumrews,
    c=actions,
    cmap="viridis",
    s=120,
    edgecolors="k",
    linewidth=0.8,
)
ax.set_xlabel("final fish biomass (sustainability proxy)")
ax.set_ylabel("cumulative reward (profit proxy)")
ax.set_title("Pareto plot — sustainability vs profit, color = action")
plt.colorbar(scatter, label="constant action a")

for label, idx, offset in [
    ("passive a=0.00", 0, (12, -10)),
    (f"optimal a={actions[i_best]:.2f}", i_best, (12, 10)),
    ("greedy a=1.00", -1, (-90, -25)),
]:
    ax.annotate(
        label,
        (finals[idx], cumrews[idx]),
        xytext=offset,
        textcoords="offset points",
        fontsize=9,
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
        arrowprops={"arrowstyle": "->", "color": "k", "alpha": 0.4},
    )
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(
    np.clip(fish_history, 0, FISH_PLOT_YMAX),
    aspect="auto",
    cmap="viridis",
    origin="lower",
    extent=[0, DEMO_HORIZON * DEMO_DT, actions[0], actions[-1]],
    vmin=0,
    vmax=FISH_PLOT_YMAX,
)
ax.set_xlabel("time")
ax.set_ylabel("constant action a")
ax.set_title("Fish biomass — action × time heatmap (clipped at 30)")
ax.axhline(
    actions[i_best],
    color="orange",
    ls="--",
    lw=2,
    alpha=0.9,
    label=f"optimal a* = {actions[i_best]:.2f}",
)
ax.legend(loc="lower right", framealpha=0.9)
plt.colorbar(im, ax=ax, label="fish biomass (clipped)")
plt.tight_layout()
plt.show()

## 5. Physical cap deep dive

When the agent demands more than is physically available, the env caps the realized harvest. Two regimes:

1. **Saturated demand**: with a very low initial stock and `action=1.0`, the cap engages immediately. The gap between demanded and realized is *wasted intent*.
2. **Greedy then back off**: the agent fishes hard for ~4 time units, then stops. Does the stock recover?

In [ ]:
params = EcologyParams(dt=DEMO_DT, fish_init=0.5, algae_init=10.0, noise_std=0.0)
env = FisheryEnv(params=params, max_harvest_rate=5.0, horizon=50)
env.reset(seed=42)

dem, real, fish = [], [], []
for _ in range(50):
    _, _, _, truncated, info = env.step(np.array([1.0], dtype=np.float32))
    dem.append(info["harvest_demanded"])
    real.append(info["harvest_realized"])
    fish.append(info["fish"])
    if truncated:
        break

t = np.arange(len(dem)) * DEMO_DT

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(t, dem, label="demanded", lw=2)
ax1.plot(t, real, label="realized (capped)", lw=2)
ax1.fill_between(t, real, dem, alpha=0.25, color="red", label="wasted intent")
ax1.set_xlabel("time")
ax1.set_ylabel("harvest")
ax1.set_title("Demand vs realized (action=1.0, low stock)")
ax1.legend()

ax2.plot(t, fish, color="C2", lw=2)
ax2.set_xlabel("time")
ax2.set_ylabel("fish biomass")
ax2.set_title("Stock under saturated demand")
plt.tight_layout()
plt.show()

In [ ]:
params = EcologyParams(dt=DEMO_DT, fish_init=10.0, algae_init=20.0, noise_std=0.0)
env = FisheryEnv(params=params, max_harvest_rate=2.0, horizon=DEMO_HORIZON)
env.reset(seed=42)

fish_hist, action_hist = [], []
switch_step = 80  # ~4 time units of greedy phase, then 16 time units of recovery
for t in range(DEMO_HORIZON):
    a = 1.0 if t < switch_step else 0.0
    _, _, _, _, info = env.step(np.array([a], dtype=np.float32))
    fish_hist.append(info["fish"])
    action_hist.append(a)

time_axis_recovery = np.arange(DEMO_HORIZON) * DEMO_DT
switch_time = switch_step * DEMO_DT

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(time_axis_recovery, action_hist, color="C3", drawstyle="steps-post", lw=2)
ax1.set_ylabel("action")
ax1.set_title("Greedy then back off — does the stock recover?")
ax1.axvspan(0, switch_time, alpha=0.15, color="red", label="greedy phase")
ax1.axvspan(
    switch_time,
    DEMO_HORIZON * DEMO_DT,
    alpha=0.15,
    color="green",
    label="recovery phase",
)
ax1.legend(loc="upper right")

ax2.plot(time_axis_recovery, fish_hist, color="C0", lw=2)
ax2.axhline(
    env.params.alpha / env.params.beta,
    color="green",
    ls=":",
    alpha=0.7,
    label=r"natural equilibrium $F^*$",
)
ax2.set_ylabel("fish biomass")
ax2.set_xlabel("time")
ax2.axvline(switch_time, color="k", ls=":", alpha=0.5)
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"Stock minimum during greedy phase: {min(fish_hist[:switch_step]):.2f}")
print(f"Stock at end of recovery:         {fish_hist[-1]:.2f}")

## 6. Sensitivity to `max_harvest_rate`

`max_harvest_rate` controls *how much* one unit of action can extract per unit of time. Different values change what the agent can do, so the **optimal policy shifts**.

We sweep `max_rate ∈ {0.5, 1.0, 2.0, 4.0}` and for each one find the action that maximizes cumulative reward.

In [ ]:
max_rates = [0.5, 1.0, 2.0, 4.0]
actions_grid = np.linspace(0.0, 1.0, 21)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for max_rate in max_rates:
    cum_per_action = []
    for a in actions_grid:
        env = make_env(max_harvest_rate=max_rate)
        env.reset(seed=42)
        action_vec = np.array([a], dtype=np.float32)
        cum = 0.0
        for _ in range(DEMO_HORIZON):
            _, reward, _, _, _ = env.step(action_vec)
            cum += reward
        cum_per_action.append(cum)
    cum_per_action = np.array(cum_per_action)
    axes[0].plot(actions_grid, cum_per_action, lw=2, label=f"max_rate = {max_rate}")
    optimal_a = actions_grid[int(np.argmax(cum_per_action))]
    axes[1].scatter([max_rate], [optimal_a], s=130, edgecolors="k")
    axes[1].annotate(
        f"a*={optimal_a:.2f}",
        (max_rate, optimal_a),
        xytext=(8, 6),
        textcoords="offset points",
        fontsize=9,
    )

axes[0].set_xlabel("constant action a")
axes[0].set_ylabel("cumulative reward")
axes[0].set_title("Reward vs action, across max_rates")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_xlabel("max_harvest_rate")
axes[1].set_ylabel("optimal action a*")
axes[1].set_title("Optimal action shifts with max_rate")
axes[1].set_xscale("log")
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

## 7. Takeaways

- **Reward concavity (D-001)** punishes bursty harvesting and rewards regularity. Visible as a saturating curve next to the linear and utility-scaled alternatives.

- **Stochasticity matters**: log-normal initial-state noise creates real spreads across seeds. Always look at a band, not a single line.

- **The action sweep makes the tragedy of the commons visible**: cumulative reward peaks at a moderate action and collapses for greedy actions because the stock vanishes and the cap shuts the harvest down. The Pareto plot shows that **more sustainability does not have to mean less profit**.

- **The physical cap is interpretable, not a hack**: demanded > available simply means the realized harvest is bounded by physics. After a greedy phase, the stock can **partially recover** if the agent backs off in time (full recovery may take much longer than the demo horizon — the system oscillates around the natural equilibrium).

- **`max_harvest_rate` is part of the env, not the agent**: changing it shifts the optimal action. Higher max-rate ⇒ smaller optimal action (because each unit of action does more damage).

This sets up **Brick 3**: a regulator will overlay a mechanism (quota, fine, ban) on top of `FisheryEnv` to shape the agent's incentives — explicitly bending the Pareto curve.